Databricks notebook source
MAGIC %md
MAGIC # Silver data quality checks (run after the Silver load)
MAGIC
MAGIC The Silver notebook validates rows **before** loading them. This notebook checks the **resulting table**:
MAGIC unique keys, the ERROR rules, consistency inside each subject, visit order, freshness and quarantine health.
MAGIC ERROR checks fail the job. WARN checks are logged only.

In [0]:
# COMMAND ----------
import os, sys, uuid

In [0]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [0]:
from pyspark.sql import Window, functions as F

In [0]:
from common import mappings as M
from common.silver_rules import RULES
from common.dq_checks import apply_rules, summarise, DQLog

In [0]:
# COMMAND ----------
dbutils.widgets.text("catalog", "clinical_dev")
dbutils.widgets.text("max_staleness_hours", "36")
dbutils.widgets.text("max_warn_row_pct", "5")
dbutils.widgets.text("max_quarantine_age_days", "7")

In [0]:
catalog = dbutils.widgets.get("catalog")
max_stale = float(dbutils.widgets.get("max_staleness_hours"))
max_warn_pct = float(dbutils.widgets.get("max_warn_row_pct"))
max_q_days = int(dbutils.widgets.get("max_quarantine_age_days"))

In [0]:
bronze_tbl = f"{catalog}.bronze.clinical_trial_raw"
silver_tbl = f"{catalog}.silver.clinical_trial"
quar_tbl = f"{catalog}.silver.clinical_trial_quarantine"
results_tbl = f"{catalog}.silver.dq_results"

In [0]:
run_id = str(uuid.uuid4())
log = DQLog(run_id, "silver_post_load")
silver = spark.table(silver_tbl)
total = silver.count()

In [0]:
# COMMAND ----------
# 1. Table-level checks
log.add("table_not_empty", "ERROR", total > 0, total, "> 0 rows")
log.add("silver_not_larger_than_bronze", "ERROR", total <= spark.table(bronze_tbl).count(), total, "<= Bronze rows")

In [0]:
dup_keys = silver.groupBy("subject_id", "visit_name").count().filter("count > 1").count()
log.add("key_unique", "ERROR", dup_keys == 0, dup_keys, "0 duplicate subject + visit keys")

In [0]:
null_keys = silver.filter("subject_id IS NULL OR visit_name IS NULL OR visit_date IS NULL").count()
log.add("key_not_null", "ERROR", null_keys == 0, null_keys, "0 rows")

In [0]:
# COMMAND ----------
# 2. Re-apply the validation rules to what is actually stored.
#    ERROR rules must find nothing (quarantine should have caught everything).
checked = apply_rules(silver, RULES)
s = summarise(checked, RULES)
log.add("error_rules_clean", "ERROR", s["error_rows"] == 0, s["error_rows"], "0 rows",
        ", ".join(f"{n}={c}" for n, sev, c in s["by_rule"] if sev == "ERROR" and c))
warn_pct = 100.0 * s["warn_rows"] / max(total, 1)
log.add("warning_row_rate_pct", "WARN", warn_pct <= max_warn_pct, warn_pct, f"<= {max_warn_pct}%")
for name, sev, c in s["by_rule"]:
    if sev == "WARN" and c:
        log.info(f"warn:{name}", c)

In [0]:
# COMMAND ----------
# 3. Consistency inside each subject (these facts must not change between visits)
multi_arm = (silver.groupBy("subject_id").agg(F.countDistinct("treatment_arm").alias("n")).filter("n > 1").count())
log.add("one_arm_per_subject", "ERROR", multi_arm == 0, multi_arm, "0 subjects")

In [0]:
multi_site = (silver.groupBy("subject_id").agg(F.countDistinct("site_id").alias("n")).filter("n > 1").count())
log.add("one_site_per_subject", "WARN", multi_site == 0, multi_site, "0 subjects")

In [0]:
no_baseline = (silver.groupBy("subject_id")
               .agg(F.max(F.when(F.col("visit_name") == "Baseline", 1).otherwise(0)).alias("b"))
               .filter("b = 0").count())
log.add("baseline_visit_present", "WARN", no_baseline == 0, no_baseline, "0 subjects without a Baseline visit")

In [0]:
# visits must be in chronological order: Screening, Baseline, Week 4, Week 8, Week 12
seq = None
for i, name in enumerate(M.VISIT_ORDER):
    seq = F.when(F.col("visit_name") == name, F.lit(i)) if seq is None else seq.when(F.col("visit_name") == name, F.lit(i))
w = Window.partitionBy("subject_id").orderBy("seq")
out_of_order = (silver.withColumn("seq", seq).withColumn("prev_date", F.lag("visit_date").over(w))
                .filter("prev_date IS NOT NULL AND visit_date < prev_date").count())
log.add("visit_dates_in_order", "WARN", out_of_order == 0, out_of_order, "0 visits")

In [0]:
# COMMAND ----------
# 4. Freshness and quarantine health
hours = silver.agg(((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.max("_silver_loaded_at"))) / 3600).alias("h")).first()["h"]
log.add("silver_freshness_hours", "WARN", hours is not None and hours <= max_stale, hours, f"<= {max_stale} hours")

In [0]:
quar = spark.table(quar_tbl)
log.info("quarantine_open_rows", quar.count(), "rows waiting for a corrected version from the source")
old_q = quar.filter(f"quarantined_at < current_timestamp() - INTERVAL {max_q_days} DAYS").count()
log.add("quarantine_not_stale", "WARN", old_q == 0, old_q, f"0 rows older than {max_q_days} days")

In [0]:
# COMMAND ----------
# Save the results, then fail the job if any ERROR check failed
res = log.to_df(spark)
res.write.mode("append").saveAsTable(results_tbl)
display(res.orderBy(F.col("status").desc(), "severity", "check_name"))

In [0]:
if log.failed_errors():
    raise Exception("Silver DQ failed: " + ", ".join(log.failed_errors()))